In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", len(files), "files,", len(dirs), "folders")
    if root.count("/") >= 4:      # don't go deeper than 4 levels
        dirs.clear()

/kaggle/input -> 0 files, 1 folders
/kaggle/input/competitions -> 0 files, 1 folders
/kaggle/input/competitions/cassava-leaf-disease-classification -> 3 files, 4 folders


In [1]:

import torch, urllib.request

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())

# Internet check (needed to download the pretrained model weights)
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10)
    print("Internet: OK")
except Exception as e:
    print("Internet: NOT working ->", e)


GPU available: True
GPU name: Tesla T4 | count: 2
Internet: OK


In [2]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch, timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.model_selection import train_test_split

# ---------- 1. Load the data and the official label names ----------
DATA_DIR = "/kaggle/input/competitions/cassava-leaf-disease-classification"
IMG_DIR = f"{DATA_DIR}/train_images"
df = pd.read_csv(f"{DATA_DIR}/train.csv")
with open(f"{DATA_DIR}/label_num_to_disease_map.json") as f:
    label_map = {int(k): v for k, v in json.load(f).items()}
print("Images:", len(df), "| Classes:", label_map)

# ---------- 2. Stratified 80/10/10 split (same seed as before) ----------
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)
print("Split sizes:", len(train_df), len(val_df), len(test_df))

# ---------- 3. Find near-duplicate photos (perceptual hash) ----------
def dhash(path):
    """Make a 64-character fingerprint of an image."""
    with Image.open(path) as img:
        img.draft("L", (80, 80))
        small = img.convert("L").resize((9, 8))
    pixels = np.asarray(small, dtype=np.int16)
    return "".join("1" if b else "0" for b in (pixels[:, 1:] > pixels[:, :-1]).flatten())

all_df = df.copy()
all_df["hash"] = [dhash(f"{IMG_DIR}/{i}") for i in tqdm(all_df["image_id"], desc="Fingerprinting")]
split_of = {**{i: "train" for i in train_df.image_id},
            **{i: "val" for i in val_df.image_id},
            **{i: "test" for i in test_df.image_id}}
all_df["split"] = all_df["image_id"].map(split_of)

bits = np.array([[int(c) for c in h] for h in all_df["hash"]], dtype=np.float32)
splits_arr = all_df["split"].values
to_drop = set()
for start in tqdm(range(0, len(bits), 1000), desc="Comparing"):
    chunk = bits[start:start + 1000]
    dist = chunk @ (1 - bits).T + (1 - chunk) @ bits.T       # Hamming distance
    rows, cols = np.where(dist <= 6)
    for r, c in zip(rows, cols):
        i = start + r
        if i < c and splits_arr[i] != splits_arr[c]:          # a pair that crosses splits
            a, b = all_df.iloc[i], all_df.iloc[c]
            if a.split == "train":   to_drop.add(b.image_id)  # keep the training copy
            elif b.split == "train": to_drop.add(a.image_id)
            else:                    to_drop.add(a.image_id if a.split == "test" else b.image_id)

print("Leaked copies removed:", len(to_drop))
val_df = val_df[~val_df["image_id"].isin(to_drop)]
test_df = test_df[~test_df["image_id"].isin(to_drop)]
print("FINAL sizes -> train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

# ---------- 4. Save the splits ----------
os.makedirs("/kaggle/working/splits", exist_ok=True)
train_df.to_csv("/kaggle/working/splits/train.csv", index=False)
val_df.to_csv("/kaggle/working/splits/val.csv", index=False)
test_df.to_csv("/kaggle/working/splits/test.csv", index=False)

# ---------- 5. Augmentation, dataset and loaders ----------
IMG_SIZE, BATCH_SIZE = 256, 32
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class CassavaDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = Image.open(f"{self.image_dir}/{row.image_id}").convert("RGB")
        return self.transform(image), int(row.label)

train_ds = CassavaDataset(train_df, IMG_DIR, train_tf)
val_ds = CassavaDataset(val_df, IMG_DIR, eval_tf)
test_ds = CassavaDataset(test_df, IMG_DIR, eval_tf)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# ---------- 6. Quick check: load one training batch ----------
check_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
start = time.time()
images, labels = next(iter(check_loader))
print("Batch of images:", images.shape, f"| loaded in {time.time() - start:.1f}s")
print("GPU available:", torch.cuda.is_available())

Images: 21397 | Classes: {0: 'Cassava Bacterial Blight (CBB)', 1: 'Cassava Brown Streak Disease (CBSD)', 2: 'Cassava Green Mottle (CGM)', 3: 'Cassava Mosaic Disease (CMD)', 4: 'Healthy'}
Split sizes: 17117 2140 2140


Comparing: 100%|██████████| 22/22 [00:03<00:00,  5.71it/s]

Leaked copies removed: 3
FINAL sizes -> train: 17117 | val: 2137 | test: 2140


Batch of images: torch.Size([32, 3, 256, 256]) | loaded in 1.3s
GPU available: True


In [ ]:
import os
print("Working folder:", os.listdir("/kaggle/working"))
print("Splits saved:", os.path.exists("/kaggle/working/splits/train.csv"))

In [ ]:
print("alive")